In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial Unicode MS'

df = pd.read_csv('googleplaystore.csv')

df['Installs'] = pd.to_numeric(df['Installs'].astype(str).str.replace(r'[+,]', '', regex=True), errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Date'] = pd.to_datetime(df['Last Updated'], errors='coerce')

def parse_size(size_str):
    if pd.isna(size_str) or size_str == 'Varies with device': return np.nan
    if 'M' in size_str: return float(size_str.replace('M', ''))
    if 'k' in size_str: return float(size_str.replace('k', '')) / 1024
    return np.nan

df['Size_MB'] = df['Size'].apply(parse_size)
df = df.dropna(subset=['App', 'Category', 'Date', 'Installs'])

mask = (
    (df['Rating'] >= 4.2) &
    (~df['App'].str.contains(r'\d', na=False, regex=True)) &
    (df['Category'].str.startswith(('T', 'P'))) &
    (df['Reviews'] > 1000) &
    (df['Size_MB'] >= 20.0) &
    (df['Size_MB'] <= 80.0)
)
filtered_df = df[mask].copy()

translation_map = {
    'TRAVEL_AND_LOCAL': 'Voyages et local',
    'PRODUCTIVITY': 'Productividad',
    'PHOTOGRAPHY': '写真'
}
filtered_df['Category'] = filtered_df['Category'].replace(translation_map)

filtered_df['Month'] = filtered_df['Date'].dt.to_period('M').dt.to_timestamp()
monthly_data = filtered_df.groupby(['Month', 'Category'])['Installs'].sum().unstack(fill_value=0).sort_index()
cumulative_data = monthly_data.cumsum()
pct_change = monthly_data.pct_change()

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()
start_time = datetime.strptime("16:00", "%H:%M").time()
end_time = datetime.strptime("18:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig, ax = plt.subplots(figsize=(14, 8))
    categories = cumulative_data.columns
    x = cumulative_data.index
    y_data = [cumulative_data[cat] for cat in categories]
    
    colors = plt.cm.tab20(np.linspace(0, 1, len(categories)))
    ax.stackplot(x, y_data, labels=categories, colors=colors, alpha=0.5)
    
    high_growth_mask = (pct_change > 0.25).any(axis=1)
    for month in high_growth_mask[high_growth_mask].index:
        ax.axvspan(month, month + pd.DateOffset(months=1), color='black', alpha=0.2)
        
    ax.set_title('Cumulative Installs by Category (Dark Bands Indicate >25% MoM Growth)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Cumulative Installs')
    ax.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print(f"Chart hidden. Current IST Time: {current_time.strftime('%I:%M %p')}.")

Chart hidden. Current IST Time: 07:34 PM.
